# TBX11K — Object Detection Pipeline

## 4 Architectures: FCOS, EfficientDet-D2, RetinaNet, DETR

This notebook runs the full pipeline:
1. Install dependencies
2. Exploratory Data Analysis
3. Convert annotations to COCO format
4. Train all 4 models (with best checkpoint selection via val mAP)
5. Generate confusion matrices + XAI explanations
6. Compare results

In [ ]:
import os
import sys
import json

KAGGLE_INPUT = '/kaggle/input'
WORK_DIR = '/kaggle/working/tb-detection'

# Copy project files to working dir if running on Kaggle
if os.path.exists(KAGGLE_INPUT):
    !cp -r $KAGGLE_INPUT/tbx11k-dataset/* {WORK_DIR}/Images/ 2>/dev/null || echo 'Images may be at different path'
    !cp -r /kaggle/input/tb-detection-pipeline/* {WORK_DIR}/ 2>/dev/null || echo 'Using existing files'

## Step 1: Install Dependencies

In [ ]:
!pip install -q torch>=2.0.0 torchvision>=0.15.0 effdet timm pycocotools pillow tqdm matplotlib seaborn
import torch
print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

## Step 2: Exploratory Data Analysis

In [ ]:
!python eda.py
from IPython.display import Image as DImg
for fname in ['tag_distribution.png', 'bbox_class_distribution.png', 'bbox_spatial_heatmap.png',
              'bbox_size_distribution.png', 'bbox_aspect_ratio.png', 'boxes_per_image.png', 'sample_grid.png']:
    path = f'results/eda/{fname}'
    if os.path.exists(path):
        display(DImg(path))

## Step 3: Convert Annotations

In [ ]:
# Adjust path if Images are at a different location
images_path = 'Images' if os.path.exists('Images') else '/kaggle/input/tbx11k-dataset/Images'
!python convert.py --data-root {images_path} --output-dir dataset
!echo 'Dataset structure:' && find dataset -maxdepth 2 -type d | head -20

## Step 4: Train FCOS

In [ ]:
!python train_fcos.py 2>&1 | tee results/fcos/training_log.txt
print('FCOS complete.')

## Step 5: Train EfficientDet-D2

In [ ]:
!python train_efficientdet.py 2>&1 | tee results/efficientdet/training_log.txt
print('EfficientDet-D2 complete.')

## Step 6: Train RetinaNet

In [ ]:
!python train_retinanet.py 2>&1 | tee results/retinanet/training_log.txt
print('RetinaNet complete.')

## Step 7: Train DETR

In [ ]:
!python train_detr.py 2>&1 | tee results/detr/training_log.txt
print('DETR complete.')

## Step 8: Model Comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

models = ['fcos', 'efficientdet', 'retinanet', 'detr']
model_labels = ['FCOS', 'EfficientDet-D2', 'RetinaNet', 'DETR']

all_metrics = {}
for model_name in models:
    path = f'results/{model_name}/metrics.json'
    if os.path.exists(path):
        with open(path) as f:
            all_metrics[model_name] = json.load(f)

df = pd.DataFrame(all_metrics).T
df.index = model_labels
print('\n=== MODEL COMPARISON ===')
print(df.round(4).to_string())
df.round(4).to_csv('results/comparison.csv')
print('\nSaved to results/comparison.csv')

In [ ]:
# Bar plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'mAP@0.5:0.95' in df.columns:
    df['mAP@0.5:0.95'].plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
    axes[0].set_ylabel('mAP@0.5:0.95')
    axes[0].set_title('Detection Performance')
    axes[0].set_ylim(0, max(df['mAP@0.5:0.95']) * 1.2)
    axes[0].grid(axis='y', alpha=0.3)

if 'AP_ActiveTB' in df.columns and 'AP_ObsoleteTB' in df.columns:
    df[['AP_ActiveTB', 'AP_ObsoleteTB']].plot(kind='bar', ax=axes[1])
    axes[1].set_ylabel('AP')
    axes[1].set_title('Per-Class Average Precision')
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].legend(['ActiveTB', 'ObsoleteTB'])

plt.tight_layout()
plt.savefig('results/comparison_barplot.png', dpi=150)
plt.show()
print('Comparison plot saved to results/comparison_barplot.png')

## Step 9: View Confusion Matrices

In [ ]:
from IPython.display import Image as DImg, display
for model_name, label in zip(models, model_labels):
    cm_path = f'results/{model_name}/confusion_matrix.png'
    if os.path.exists(cm_path):
        print(f'\n{label}:')
        display(DImg(cm_path))

## Step 10: View XAI Explanations

In [ ]:
from IPython.display import Image as DImg, display
for model_name, label in zip(models, model_labels):
    explain_dir = f'results/{model_name}/explain'
    if os.path.isdir(explain_dir):
        samples = sorted(os.listdir(explain_dir))[:3]
        for s in samples:
            display(DImg(os.path.join(explain_dir, s)))

---
## Summary

All 4 models trained and evaluated. Results saved to `results/`.

| Model | mAP@0.5:0.95 | mAP@0.5 | AP_ActiveTB | AP_ObsoleteTB |
|-------|-------------|---------|-------------|--------------|
| FCOS | ... | ... | ... | ... |
| EfficientDet-D2 | ... | ... | ... | ... |
| RetinaNet | ... | ... | ... | ... |
| DETR | ... | ... | ... | ... |